In [0]:
#importing req functions
from pyspark.sql.functions import *
from pyspark.sql.types import *

# Silver Vehicle master

In [0]:
#vehicla master transformation
vehicle_master = spark.read.table("proj_veh.bronze.vehicle_master")

silver_vehicle_master1 = vehicle_master\
                                        .withColumn('year',col('year').cast(IntegerType()))\
                                        .withColumn('purchase_cost_usd',col('purchase_cost_usd').cast(DoubleType()))\
                                        .withColumn('first_service_date',col('first_service_date').cast(DateType()))\
                                        .withColumn('fleet_entry_date',col('fleet_entry_date').cast(DateType()))\
                                        .withColumn('ingestion_timestamp',current_timestamp())\
                                        .withColumn('make',initcap(col('make')))\
                                        .withColumn('model',initcap(col('model')))

# checking null before dropping

silver_vehicle_master1.select([count(when(col(column).isNull(), column)).alias(column) for column in silver_vehicle_master1.columns]).show()

silver_vehicle_master = silver_vehicle_master1\
                        .dropna(subset =["first_service_date"])

#writing to silver
silver_vehicle_master.write.mode('overwrite').format('delta').saveAsTable('proj_veh.silver.silver_vehicle_master')

+----------+----+-----+----+------------------+-----------------+----------------+-------------------+
|vehicle_id|make|model|year|first_service_date|purchase_cost_usd|fleet_entry_date|ingestion_timestamp|
+----------+----+-----+----+------------------+-----------------+----------------+-------------------+
|         0|   0|    0|   0|                 1|                0|               0|                  0|
+----------+----+-----+----+------------------+-----------------+----------------+-------------------+



# Silver oem_recalls

In [0]:
oem_recalls = spark.read.table("proj_veh.bronze.oem_recalls")

silver_oem_recalls = oem_recalls\
    .withColumn('year_start',col('year_start').cast(IntegerType()))\
    .withColumn('year_end',col('year_end').cast(IntegerType()))\
    .withColumn('recall_date',col('recall_date').cast(DateType()))\
    .withColumn('ingestion_timestamp',current_timestamp())


silver_oem_recalls.write.mode('overwrite').format('delta').saveAsTable('proj_veh.silver.silver_oem_recalls')
    

# Silver maintenance_logs

In [0]:
maintenance_logs =  spark.read.table("proj_veh.bronze.maintenance_logs")


silver_maintenance_logs = maintenance_logs\
    .withColumn("service_date",col("service_date").cast(DateType()))\
    .withColumn("mileage_km",col("mileage_km").cast(DoubleType()))\
    .withColumn("cost_usd",col("cost_usd").cast(DoubleType()))\
    .withColumn("service_type",trim(col("service_type")))\
    .withColumn("part_replaced",trim(col("part_replaced")))\
    .withColumn('ingestion_timestamp',current_timestamp())



silver_maintenance_logs.write.mode('overwrite').format('delta').saveAsTable('proj_veh.silver.silver_maintenance_logs')

# Silver warranty claims

In [0]:
warranty_claims = spark.read.table("proj_veh.bronze.warranty_claims")

silver_warranty_claims = warranty_claims\
    .withColumn('claim_date',col('claim_date').cast(DateType()))\
    .withColumn('service_date_ref',col('service_date_ref').cast(DateType()))\
    .withColumn('claim_amount_usd',col('claim_amount_usd').cast(DoubleType()))\
    .withColumn('ingestion_timestamp',current_timestamp())


silver_warranty_claims.write.mode('overwrite').format('delta').saveAsTable('proj_veh.silver.silver_warranty_claims')